In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/jvkrishwanth/extracted-d1/df_main_phase1.csv
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_27_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_14_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_3_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_22_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_15_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_16_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_23_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_19_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)/P_7_MW-clean-epo

In [3]:
"""Kaggle-ready EEG band-power analysis for Focus versus Mind Wandering.

Outputs
-------
* One trajectory plot each for Delta, Theta, Alpha, Beta and Gamma.
* One combined trajectory plot (band-wise standardised log power).
* Tests of MW-vs-Focus mean power for every band.
* Tests of whether each band increases/decreases over time in
  Focus and in MW.

Attach the FIF files and the labels CSV in Kaggle, then edit FIF_DIR and
LABEL_CSV below. All outputs are written to /kaggle/working/all_band_results.
"""
import os
from pathlib import Path
import warnings

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from scipy.stats import norm
import statsmodels.formula.api as smf

# Auto-detect local vs Kaggle paths
KAGGLE_FIF = Path("/kaggle/input/datasets/jvkrishwanth/extracted-d1/extracted_dataset_ICA_CLEANED (1)")
LOCAL_FIF = Path("D:/downloads/final_exg/ICA_cleaned")
FIF_DIR = LOCAL_FIF if LOCAL_FIF.exists() else KAGGLE_FIF

KAGGLE_LABEL = Path("/kaggle/input/datasets/jvkrishwanth/extracted-d1/df_main_phase1.csv")
LOCAL_LABEL = Path("D:/downloads/final_exg_cleaned_all_phases/df_main_phase1.csv")
LABEL_CSV = LOCAL_LABEL if LOCAL_LABEL.exists() else KAGGLE_LABEL

FIF_TEMPLATE = "P_{subject}_MW-clean-epo.fif"
OUTPUT_DIR = Path("output") if Path("output").exists() else Path("/kaggle/working/all_band_results")
CSV_DIR = OUTPUT_DIR / "csv files" if (OUTPUT_DIR / "csv files").exists() else OUTPUT_DIR
PLOTS_DIR = OUTPUT_DIR / "plots" if (OUTPUT_DIR / "plots").exists() else OUTPUT_DIR

BANDS = {
    "Delta": (1.0, 4.0), "Theta": (4.0, 8.0), "Alpha": (8.0, 12.0),
    "Beta": (12.0, 30.0), "Gamma": (30.0, 45.0),
}
WINDOW_SECONDS, STEP_SECONDS = 1.0, 0.5
CHANNELS = ["FP1", "FP2", "F7", "F3", "FZ", "F4", "F8", "T3", "C3", "CZ", "C4",
            "T4", "T5", "P3", "PZ", "P4", "T6", "O1", "O2"]
MW_EPOCH_TMIN = -5.0
FOCUS_START_OFFSET = 1.0


def normalise_state(values):
    value = values.astype("string").str.strip().str.upper()
    result = pd.Series(np.where(value.isin({"MW", "MIND WANDERING", "MIND_WANDERING", "1"}),
                                "MW", "Focus"), index=values.index, dtype="string")
    result.loc[value.isna() | value.eq("")] = pd.NA
    return result


def labels_for_subject(labels, subject, n_epochs):
    required = {"Subject", "State"}
    if not required.issubset(labels.columns):
        raise ValueError(f"LABEL_CSV must include {required}; found {list(labels.columns)}")
    sub = labels.loc[labels["Subject"].astype(str) == str(subject)].copy()
    order = [c for c in ["Session", "Epoch_Index"] if c in sub.columns]
    if order:
        sub = sub.sort_values(order).groupby(order, as_index=False)["State"].first()
    state = normalise_state(sub["State"]).to_numpy()
    if len(state) != n_epochs:
        warnings.warn(f"Skipping subject {subject}: {len(state)} labels versus {n_epochs} EEG epochs.")
        return np.array([])
    return state


def extract_windows(labels):
    if not FIF_DIR.exists():
        existing_csv = CSV_DIR / "all_band_window_level_data.csv"
        if existing_csv.exists():
            print(f"Reading existing window data from {existing_csv}")
            return pd.read_csv(existing_csv)
        raise RuntimeError("Neither FIF_DIR nor existing CSV was found.")
    rows = []
    fif_files = sorted(FIF_DIR.glob("P_*_MW*-epo.fif"))
    for fif_path in fif_files:
        subject = fif_path.name.split("_")[1]
        expected = FIF_DIR / FIF_TEMPLATE.format(subject=subject)
        if expected.exists() and fif_path != expected:
            continue
        print(f"Reading {fif_path.name}")
        epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
        available = [c for c in CHANNELS if c in epochs.ch_names]
        if not available:
            continue
        epochs.pick(available)
        state = labels_for_subject(labels, subject, len(epochs))
        if not len(state):
            state = ["MW" if event[2] in (1, 101) else "Focus" for event in epochs.events]
        if not len(state):
            continue
        sfreq = epochs.info["sfreq"]
        window_n, step_n = round(WINDOW_SECONDS * sfreq), round(STEP_SECONDS * sfreq)
        data = epochs.get_data(copy=True)
        pad_samples = round(5.0 * sfreq) - data.shape[-1]
        if pad_samples > 0:
            data = np.pad(data, ((0, 0), (0, 0), (0, pad_samples)), mode="edge")

        for epoch_index, (epoch, epoch_state) in enumerate(zip(data, state)):
            if pd.isna(epoch_state):
                continue
            for window_index, start in enumerate(range(0, epoch.shape[-1] - window_n + 1, step_n)):
                # Do NOT omit window 8 in MW; length of MW and Focus must be identical
                segment = epoch[:, start:start + window_n]
                freqs, psd = welch(segment, fs=sfreq, nperseg=window_n, axis=-1)
                row = {"Subject": str(subject), "Epoch": f"{subject}_{epoch_index}", "State": epoch_state,
                       "Epoch_Index": epoch_index, "Window_Index": window_index,
                       "Time_s": (start + window_n / 2) / sfreq}
                for band, (low, high) in BANDS.items():
                    mask = (freqs >= low) & (freqs < high)
                    if mask.any():
                        row[f"Log10_{band}_Power"] = np.log10(psd[:, mask].mean() + 1e-20)
                rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No windows extracted. Check FIF_DIR, FIF_TEMPLATE, channel names, and labels.")
    return df


def epoch_summaries(windows, band):
    power_col = f"Log10_{band}_Power"
    rows = []
    for (subject, epoch, state), part in windows.groupby(["Subject", "Epoch", "State"], observed=True):
        part = part.dropna(subset=["Time_s", power_col])
        if part.empty:
            continue
        row = {"Subject": subject, "Epoch": epoch, "State": state,
               "Mean_log10_power": part[power_col].mean(), "N_Windows": len(part)}
        if len(part) >= 3 and part["Time_s"].nunique() >= 2:
            row["Slope_log10_power_per_second"] = np.polyfit(part["Time_s"], part[power_col], 1)[0]
        else:
            row["Slope_log10_power_per_second"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def contrast_result(fit, contrast, label, effect, alternative="two-sided"):
    params, cov = fit.params, fit.cov_params().loc[fit.params.index, fit.params.index]
    estimate = float(contrast @ params.to_numpy())
    se = float(np.sqrt(contrast @ cov.to_numpy() @ contrast))
    z = estimate / se
    p = 2 * norm.sf(abs(z)) if alternative == "two-sided" else (norm.sf(z) if alternative == ">" else norm.cdf(z))
    return {"Band": label, "Analysis": effect, "Estimate": estimate, "SE": se, "z": z,
            "CI_95_low": estimate - 1.96 * se, "CI_95_high": estimate + 1.96 * se,
            "p_value": p}


def analyse_band(windows, band):
    summary = epoch_summaries(windows, band)
    summary["State"] = pd.Categorical(summary["State"], categories=["Focus", "MW"])
    if not {"Focus", "MW"}.issubset(set(summary["State"].dropna())):
        raise ValueError(f"{band}: both Focus and MW epochs are required.")

    # MW - Focus mean epoch power. Clustered SEs account for repeated epochs per participant.
    mean_fit = smf.ols("Mean_log10_power ~ C(State, Treatment(reference='Focus'))", summary).fit(
        cov_type="cluster", cov_kwds={"groups": summary["Subject"]})
    mean_names = list(mean_fit.params.index)
    state_term = next(n for n in mean_names if n.startswith("C(State,"))
    mean_contrast = np.zeros(len(mean_names)); mean_contrast[mean_names.index(state_term)] = 1
    results = [contrast_result(mean_fit, mean_contrast, band, "MW minus Focus mean power")]

    # State-specific epoch slopes. A two-sided p value tests change in either direction;
    # the estimate's sign reports whether that change is increase or decrease.
    trend = summary.dropna(subset=["Slope_log10_power_per_second"]).copy()
    trend_fit = smf.ols("Slope_log10_power_per_second ~ C(State, Treatment(reference='Focus'))", trend).fit(
        cov_type="cluster", cov_kwds={"groups": trend["Subject"]})
    names = list(trend_fit.params.index)
    state_term = next(n for n in names if n.startswith("C(State,"))
    focus = np.zeros(len(names)); focus[names.index("Intercept")] = 1
    mw = focus.copy(); mw[names.index(state_term)] = 1
    results.extend([
        contrast_result(trend_fit, focus, band, "Focus change over time"),
        contrast_result(trend_fit, mw, band, "MW change over time"),
    ])
    return summary, results


def add_statistical_decisions(results):
    """Annotate each test at alpha = 0.05."""
    results = results.copy()
    decisions = []
    for _, row in results.iterrows():
        if row["p_value"] >= 0.05:
            decisions.append("Not statistically significant")
        elif row["Analysis"] == "MW minus Focus mean power":
            decisions.append("MW higher power" if row["Estimate"] > 0 else "MW lower power")
        else:
            decisions.append("Increases over time" if row["Estimate"] > 0 else "Decreases over time")
    results["Statistical_decision_alpha_0.05"] = decisions
    return results


def plot_band(windows, band):
    value = f"Log10_{band}_Power"
    subj = windows.groupby(["Subject", "State", "Time_s"], as_index=False)[value].mean()
    stats = subj.groupby(["State", "Time_s"], as_index=False)[value].agg(mean="mean", sem="sem")
    fig, ax = plt.subplots(figsize=(10, 6))
    for state, colour in [("Focus", "#2878B5"), ("MW", "#D1495B")]:
        s = stats[stats.State == state]
        plot_time = s.Time_s + MW_EPOCH_TMIN
        ax.plot(plot_time, s["mean"], marker="o", lw=3.0, markersize=8, label=state, color=colour)
        ax.fill_between(plot_time, s["mean"] - s["sem"], s["mean"] + s["sem"], color=colour, alpha=0.2)
    ax.set_title(f"{band}-band power over time", fontsize=22, fontweight="bold", pad=14)
    ax.set_xlabel("Time (s)", fontsize=20, fontweight="bold", labelpad=10)
    ax.set_ylabel(f"Mean log10 {band.lower()} PSD (power/Hz)", fontsize=20, fontweight="bold", labelpad=10)
    ax.set_xticks([-4.5, -3.5, -2.5, -1.5, -0.5])
    band_ylims = {
        "Delta": (-11.55, -11.10),
        "Theta": (-12.15, -11.85),
        "Alpha": (-12.35, -11.45),
        "Beta":  (-12.70, -12.30),
        "Gamma": (-13.35, -12.85),
    }
    band_yticks = {
        "Delta": [-11.5, -11.4, -11.3, -11.2, -11.1],
        "Theta": [-12.1, -12.0, -11.9],
        "Alpha": [-12.3, -12.1, -11.9, -11.7, -11.5],
        "Beta":  [-12.7, -12.6, -12.5, -12.4, -12.3],
        "Gamma": [-13.3, -13.2, -13.1, -13.0, -12.9],
    }
    if band in band_ylims:
        ax.set_ylim(band_ylims[band])
        ax.set_yticks(band_yticks[band])
    ax.tick_params(axis="both", labelsize=16)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")
    ax.grid(True, color="#c7c7c7", linewidth=1.1)
    for spine in ax.spines.values():
        spine.set_edgecolor("#c7c7c7")
        spine.set_linewidth(1.1)
    leg = ax.legend(title="State", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=16, title_fontsize=18)
    leg.get_title().set_fontweight("bold")
    for t in leg.get_texts():
        t.set_fontweight("bold")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{band.lower()}_band_power_over_time.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    return subj


def plot_all_bands(subject_curves):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), sharey=True)
    colours = {"Delta": "#4C78A8", "Theta": "#F58518", "Alpha": "#54A24B", "Beta": "#E45756", "Gamma": "#B279A2"}
    for ax, state in zip(axes, ["Focus", "MW"]):
        for band, curve in subject_curves.items():
            value = f"Log10_{band}_Power"
            part = curve[curve.State == state].copy()
            z = (part[value] - subject_curves[band][value].mean()) / subject_curves[band][value].std(ddof=0)
            summary = pd.DataFrame({"Time_s": part.Time_s, "z": z}).groupby("Time_s").z.agg(["mean", "sem"]).reset_index()
            plot_time = summary.Time_s + MW_EPOCH_TMIN
            ax.plot(plot_time, summary["mean"], marker="o", lw=2.5, markersize=7, label=band, color=colours[band])
            ax.fill_between(plot_time, summary["mean"] - summary["sem"], summary["mean"] + summary["sem"], color=colours[band], alpha=0.15)
        ax.axhline(0, color="black", lw=1.0)
        ax.set_title(state, fontsize=22, fontweight="bold", pad=12)
        ax.set_xlabel("Time relative to bell (s)", fontsize=20, fontweight="bold", labelpad=10)
        ax.set_xticks([-4, -3, -2, -1])
        ax.tick_params(axis="both", labelsize=16)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight("bold")
        ax.grid(True, color="#c7c7c7", linewidth=1.1)
        for spine in ax.spines.values():
            spine.set_edgecolor("#c7c7c7")
            spine.set_linewidth(1.1)
    axes[0].set_ylim(-0.4, 0.4)
    axes[0].set_yticks([-0.4, -0.2, 0.0, 0.2, 0.4])
    axes[0].set_ylabel("Standardised mean log10 PSD (z)", fontsize=20, fontweight="bold", labelpad=10)
    for label in axes[0].get_yticklabels():
        label.set_fontweight("bold")
    leg = axes[1].legend(title="Band", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=16, title_fontsize=18)
    leg.get_title().set_fontweight("bold")
    for t in leg.get_texts():
        t.set_fontweight("bold")
    fig.suptitle("All EEG bands: power trajectories", y=1.03, fontsize=24, fontweight="bold")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "all_bands_power_over_time_standardised.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def main():
    CSV_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", context="talk")
    windows = extract_windows(pd.read_csv(LABEL_CSV))
    windows.to_csv(CSV_DIR / "all_band_window_level_data.csv", index=False)
    all_results, curves = [], {}
    for band in BANDS:
        summary, band_results = analyse_band(windows, band)
        summary.to_csv(CSV_DIR / f"{band.lower()}_epoch_summaries.csv", index=False)
        all_results.extend(band_results)
        curves[band] = plot_band(windows, band)
    results = add_statistical_decisions(pd.DataFrame(all_results))
    results.to_csv(CSV_DIR / "all_band_statistical_results.csv", index=False)
    plot_all_bands(curves)
    print("\nStatistical results:\n", results.to_string(index=False))
    significant_results = results.loc[results["p_value"] < 0.05].sort_values("p_value").copy()
    print("\nAll statistically significant results (p < 0.05):\n", significant_results.to_string(index=False))
    remaining_results = results.loc[results["p_value"] >= 0.05].nsmallest(5, "p_value")
    print("\nFive lowest p-value results among the remaining analyses:\n", remaining_results.to_string(index=False))    
    print(f"\nSaved plots to {PLOTS_DIR} and CSV files to {CSV_DIR}")


if __name__ == "__main__":
    main()

Reading P_10_MW-clean-epo.fif
Reading P_11_MW-clean-epo.fif
Reading P_12_MW-clean-epo.fif
Reading P_13_MW-clean-epo.fif
Reading P_14_MW-clean-epo.fif
Reading P_15_MW-clean-epo.fif
Reading P_16_MW-clean-epo.fif
Reading P_17_MW-clean-epo.fif
Reading P_18_MW-clean-epo.fif
Reading P_19_MW-clean-epo.fif
Reading P_1_MW-clean-epo.fif
Reading P_20_MW-clean-epo.fif
Reading P_22_MW-clean-epo.fif
Reading P_23_MW-clean-epo.fif
Reading P_24_MW-clean-epo.fif
Reading P_25_MW-clean-epo.fif
Reading P_26_MW-clean-epo.fif
Reading P_27_MW-clean-epo.fif
Reading P_28_MW-clean-epo.fif
Reading P_2_MW-clean-epo.fif
Reading P_3_MW-clean-epo.fif
Reading P_5_MW-clean-epo.fif
Reading P_7_MW-clean-epo.fif
Reading P_8_MW-clean-epo.fif
Reading P_9_MW-clean-epo.fif

Statistical results:
  Band                  Analysis  Estimate       SE         z  CI_95_low  CI_95_high  p_value Statistical_decision_alpha_0.05
Delta MW minus Focus mean power  0.043475 0.036692  1.184847  -0.028442    0.115392 0.236078   Not statistica